In [2]:
import numpy as np
from datetime import datetime, timedelta

In [5]:
def calculate_event_weight_ny_cut(vol_before_event, vol_after_event):
    """
    Calculate event weight for NY cut options
    
    vol_before_event: Implied vol for expiry BEFORE event (e.g., Nov 18 10am)
    vol_after_event: Implied vol for expiry AFTER event (e.g., Nov 19 10am)
    
    Both are 1-day options from market's perspective
    """
    
    # Convert to variance (variance is additive)
    var_before = vol_before_event**2 * 1  # 1 day
    var_after = vol_after_event**2 * 1   # 1 day
    
    # Event variance is the extra variance
    event_var = var_after - var_before
    
    # Event weight = how much of tomorrow's variance is from the event
    event_weight = event_var / var_after
    
    # Implied event vol (approximate daily move)
    event_vol = np.sqrt(event_var)
    
    print(f"Event Weight Analysis:")
    print(f"  Nov 18 10am vol: {vol_before_event*100:.2f}%")
    print(f"  Nov 19 10am vol: {vol_after_event*100:.2f}%")
    print(f"  Event weight: {event_weight*100:.1f}%")
    print(f"  Implied event vol: {event_vol*100:.2f}%")
    
    return event_weight, event_vol

# Example
event_weight, event_vol = calculate_event_weight_ny_cut(
    vol_before_event=0.0699,  
    vol_after_event=0.065    # 12% for Nov 19 (includes CPI)
)

Event Weight Analysis:
  Nov 18 10am vol: 6.99%
  Nov 19 10am vol: 6.50%
  Event weight: -15.6%
  Implied event vol: nan%


C:\Users\ntaylor\AppData\Local\Temp\ipykernel_35180\1183383845.py:22: RuntimeWarning: invalid value encountered in sqrt
  event_vol = np.sqrt(event_var)


In [4]:
import pdblp

import pandas as pd
import numpy as np
import pytz
from typing import Dict, Tuple
import seaborn as sns
import matplotlib.pyplot as plt

from xbbg import blp
from datetime import datetime, timedelta

In [39]:
def get_DailySpot(currency_pairs, years):
    df_ccy = {}
    start_date = (datetime.today() - timedelta(days= years * 356)).strftime('%Y-%m-%d')
    end_date = datetime.today().strftime('%Y-%m-%d')
    for ticker in currency_pairs:
        data_ccy = blp.bdh(
            tickers=f"{ticker}" ,
            flds=["PX_LAST"], 
            start_date=start_date,
            end_date=end_date,
            Per="D")
        data_ccy.columns = [ticker]
        df_ccy[ticker] = data_ccy
    df_ccyAll = pd.concat(df_ccy, axis=1)
    df_ccyAll.columns = df_ccyAll.columns.droplevel(0) if isinstance(df_ccyAll.columns, pd.MultiIndex) else df_ccyAll.columns
    df_ccyAll = df_ccyAll.sort_index(ascending=True)
    return df_ccyAll

def calculate_HistCorr_3mPLUS(currency_pairs, 
                                     years=5):
    df_prices = get_DailySpot(currency_pairs, years)
    df_returns = np.log(df_prices / df_prices.shift(1)).dropna()
    periods = {
        '3M': 63,
        '6M': 126,
        '1Y': 252,
        '2Y': 504,
        '3Y': 756,
        '5Y': 1260}
    correlations = {}
    for period_name, days in periods.items():
        if len(df_returns) >= days:
            recent_returns = df_returns.tail(days)
            corr = recent_returns.iloc[:, 0].corr(recent_returns.iloc[:, 1])
            correlations[period_name] = corr
        else:
            correlations[period_name] = np.nan
    return correlations, df_returns

def get_DailyVols(ccys, tenors):
    end_date = datetime.now().strftime("%Y-%m-%d")
    start_date = (datetime.now() - timedelta(days=1)).strftime("%Y-%m-%d")
    df_vols = {}
    for ccy in ccys:
        for tenor in tenors:
            ticker_IV = f"{ccy}V{tenor} BGN Curncy"
            data_IV = blp.bdh(
                tickers=ticker_IV,
                flds="PX_LAST",
                start_date=start_date,
                end_date=end_date)
            if not data_IV.empty:
                column_name = f"{ccy}_{tenor}"
                data_IV.columns = [column_name]
                df_vols[column_name] = data_IV
            else:
                print(f"No data for {ticker_IV}, skipping.")
    df_vols_all = pd.concat(df_vols.values(), axis=1)
    return df_vols_all



def calculate_implied_correlation(df_vols, major1, major2, cross, tenors):
    results = {}
    for tenor in tenors:
        col_major1 = f"{major1}_{tenor}"
        col_major2 = f"{major2}_{tenor}"
        col_cross = f"{cross}_{tenor}"
        if all(col in df_vols.columns for col in [col_major1, col_major2, col_cross]):
            vol_major1 = df_vols[col_major1]
            vol_major2 = df_vols[col_major2]
            vol_cross = df_vols[col_cross]
            # Calculate implied correlation
            # ρ = (σ₁² + σ₂² - σ_cross²) / (2 × σ₁ × σ₂)
            implied_corr = (vol_major1**2 + vol_major2**2 - vol_cross**2) / (2 * vol_major1 * vol_major2)
            results[tenor] = round(implied_corr, 4)
        else:
            print(f"Missing data for tenor {tenor}")
    df_corr = pd.concat(results, axis=1)
    df_corr.columns = [f"{major1}_{major2}_{tenor}" for tenor in tenors]
    return df_corr

In [40]:
ccys = ['USDJPY', 'USDCHF', 'CHFJPY']
tenors = ['1W', '1M', '2M', '3M']


df_vols = get_DailyVols(ccys, tenors)
df_implied_corr = calculate_implied_correlation(df_vols, ccys[0], ccys[1], ccys[2], tenors)


In [41]:
df_implied_corr

,USDJPY_USDCHF_1W,USDJPY_USDCHF_1M,USDJPY_USDCHF_2M,USDJPY_USDCHF_3M
2025-11-17,0.636,0.6185,0.5937,0.5852


In [42]:
def getintradayCCY(ticker, days, interv):
    con = pdblp.BCon(debug=False, port=8194, timeout=5000)
    con.start()
    eastern = pytz.timezone("US/Eastern")
    end_time_et = eastern.localize(datetime.now())
    start_time_et = end_time_et - timedelta(days=days)
    end_time_gmt = end_time_et.astimezone(pytz.utc)
    start_time_gmt = start_time_et.astimezone(pytz.utc)
    df = con.bdib(
            ticker=f"{ticker} Curncy",     
            start_datetime=start_time_gmt,    
            end_datetime=end_time_gmt,        
            event_type="TRADE",           
            interval= interv)
    df.index = df.index.tz_localize('UTC').tz_convert('US/Eastern')
    df = df.reset_index()
    df['date'] = df['time'].dt.date  # Extract the date
    df['time_close'] = df['time'].dt.time  # Extract the time
    df = df.drop(columns=['time'])
    df = df[['date', 'time_close'] + [col for col in df.columns if col not in ['date', 'time_close']]]
    return df


def calculate_realized_correlation(ccy1, ccy2, days, interv='60'):
    df1 = getintradayCCY(ccy1, days, interv)
    df2 = getintradayCCY(ccy2, days, interv)
    df1 = df1.rename(columns={'close': f'{ccy1}_close'})
    df2 = df2.rename(columns={'close': f'{ccy2}_close'})
    df_merged = pd.merge(
        df1[['date', 'time_close', f'{ccy1}_close']], 
        df2[['date', 'time_close', f'{ccy2}_close']], 
        on=['date', 'time_close'],
        how='inner')
    df_merged[f'{ccy1}_return'] = np.log(df_merged[f'{ccy1}_close'] / df_merged[f'{ccy1}_close'].shift(1))
    df_merged[f'{ccy2}_return'] = np.log(df_merged[f'{ccy2}_close'] / df_merged[f'{ccy2}_close'].shift(1))
    df_merged = df_merged.dropna()
    windows = {
        '1W': 7,
        '2W': 14,
        '1M': 30,
        '2M': 60}
    correlations = {}
    for tenor, window_days in windows.items():
        cutoff_date = df_merged['date'].max() - timedelta(days=window_days)
        df_window = df_merged[df_merged['date'] > cutoff_date].copy()
        if len(df_window) > 1: 
            corr = df_window[[f'{ccy1}_return', f'{ccy2}_return']].corr().iloc[0, 1]
            correlations[tenor] = corr
        else:
            correlations[tenor] = np.nan
    df_corr = pd.DataFrame([correlations], index=[f'{ccy1}_{ccy2}'])
    
    return df_merged, df_corr

In [43]:
df_merged, df_corr = calculate_realized_correlation('USDJPY', 'USDCHF', days=70, interv='60')

In [38]:
df_corr

,1W,2W,1M,2M
USDJPY_USDCHF,0.679422,0.606555,0.584823,0.552563
